# 02 — Feature Engineering & Split — Modelo A

Neste notebook fazemos apenas a preparação final dos dados para a fase de modelação.

O objetivo escolhido é o **Modelo A — avaliação de posts existentes**. Isto significa que o modelo não tenta prever o sucesso de um post antes da publicação. Ele aprende a aproximar um score académico de discoverability usando sinais observáveis de posts já publicados.

Pontos importantes para defender o trabalho:

- `reactions`, `comments` e `time_spent_days` podem ficar nas features dos posts porque estamos a avaliar posts existentes.
- Este notebook **não treina modelos**.
- Este notebook **não faz fit** de `StandardScaler`, `OneHotEncoder`, embeddings, imputers ou pipelines.
- O que fica guardado são apenas os dados separados em treino, validação e teste, mais metadata para reprodutibilidade.
- O preprocessamento com `fit` deve acontecer no notebook seguinte, usando apenas o conjunto de treino.

> Nota para o relatório: o score é uma proxy académica de discoverability, não o algoritmo real do LinkedIn.

In [ ]:
# ==============================
# 1. Imports e configuração base
# ==============================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


def infer_project_root():
    """Tenta encontrar a raiz do projeto.

    Em alguns projetos o notebook está dentro da pasta notebooks/.
    Nesse caso a raiz costuma ser "..".
    Se o notebook estiver diretamente na raiz, usamos ".".
    """
    candidates = [Path("..").resolve(), Path(".").resolve()]

    for root in candidates:
        if (root / "data" / "processed").exists():
            return root

    # Fallback: mantém o comportamento comum de notebooks dentro de /notebooks.
    return Path("..").resolve()


PROJECT_ROOT = infer_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Guardamos os splits numa pasta própria para não misturar com dados processados nem com modelos.
SPLITS_DIR = DATA_DIR / "splits"
TRAIN_DIR = SPLITS_DIR / "train"
VAL_DIR = SPLITS_DIR / "validation"
TEST_DIR = SPLITS_DIR / "test"
METADATA_DIR = SPLITS_DIR / "metadata"

for path in [TRAIN_DIR, VAL_DIR, TEST_DIR, METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MODEL_MODE = "A_existing_posts_evaluation"

print("Raiz do projeto:", PROJECT_ROOT)
print("Dados processados:", PROCESSED_DIR)
print("Pasta dos splits:", SPLITS_DIR)
print("Modo:", MODEL_MODE)

## 2. Carregar dados processados

Este notebook assume que o notebook anterior já guardou os datasets processados:

- `processed_profiles.pkl`
- `processed_posts.pkl`

Também fazemos `reset_index(drop=True)` para evitar problemas com índices duplicados importados de CSV.

In [ ]:
# ================================
# 2. Carregar dados processados
# ================================

profiles_path = PROCESSED_DIR / "processed_profiles.pkl"
posts_path = PROCESSED_DIR / "processed_posts.pkl"

if not profiles_path.exists():
    raise FileNotFoundError(f"Não encontrei o ficheiro: {profiles_path}")

if not posts_path.exists():
    raise FileNotFoundError(f"Não encontrei o ficheiro: {posts_path}")


df_profiles = pd.read_pickle(profiles_path).reset_index(drop=True)
df_posts = pd.read_pickle(posts_path).reset_index(drop=True)

print("Perfis:", df_profiles.shape)
print("Posts:", df_posts.shape)

print("\nColunas dos perfis:")
print(df_profiles.columns.tolist())

print("\nColunas dos posts:")
print(df_posts.columns.tolist())

## 3. Definir targets e features

Aqui definimos as colunas que podem entrar no split.

Isto ainda **não é treino** e ainda **não é preprocessamento fitted**. Estamos só a decidir que colunas fazem sentido levar para a fase seguinte.

### Posts — Modelo A

Como escolhemos avaliar posts existentes, mantemos sinais pós-publicação:

- `reactions`
- `comments`
- `time_spent_days`

Isto seria mau para previsão antes de publicar, mas está certo para avaliação de posts já publicados.

In [ ]:
# ==========================================
# 3. Definição controlada de features/targets
# ==========================================

PROFILE_TARGET = "profile_discoverability_score"
POST_TARGET = "post_discoverability_score"

# Features candidatas de perfis.
# Se alguma coluna tiver sido removida no notebook anterior, ela será simplesmente ignorada.
PROFILE_NUMERIC_CANDIDATES = [
    "connections",
    "years_experience",
    "about_length",
    "headline_length",
    "num_skills",
    "num_experience",
    "num_education",
    "num_goals",
    "num_needs",
    "num_can_offer",
]

PROFILE_CATEGORICAL_CANDIDATES = [
    "seniority_level",
    "industry",
    "remote_preference",
]

# Features candidatas de posts para o MODELO A.
# reactions/comments/time_spent_days ficam porque o objetivo é avaliar posts existentes.
# hashtag_followers não entra porque foi identificado como constante/sem informação útil.
POST_NUMERIC_CANDIDATES = [
    "followers",
    "reactions",
    "comments",
    "num_hashtags",
    "content_length",
    "num_links",
    "has_media",
    "time_spent_days",
]

POST_CATEGORICAL_CANDIDATES = [
    "media_type",
    "hashtags_bin",
]

PROFILE_TEXT_COLUMNS = ["headline", "about"]
POST_TEXT_COLUMNS = ["content"]


def require_columns(df, required, name):
    """Garante que as colunas obrigatórias existem."""
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name}: faltam colunas obrigatórias: {missing}")


def prepare_target(df, target_col, name):
    """Garante que o target é numérico e não tem valores em falta."""
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    if df[target_col].isna().any():
        n_missing = int(df[target_col].isna().sum())
        raise ValueError(f"{name}: o target {target_col} tem {n_missing} valores em falta.")


def ensure_text_columns(df, columns):
    """Cria colunas textuais opcionais caso faltem.

    Isto não aprende nada com os dados. É apenas limpeza determinística.
    """
    for col in columns:
        if col not in df.columns:
            df[col] = "Unknown"
        df[col] = df[col].fillna("Unknown").astype(str)


def prepare_numeric_candidates(df, columns):
    """Converte colunas numéricas candidatas sem aplicar scaler/imputer fitted."""
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)


def prepare_categorical_candidates(df, columns):
    """Normaliza categorias em falta sem aprender encoding."""
    for col in columns:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown").astype(str)


def select_usable_features(df, candidates, name, drop_constant=True):
    """Seleciona features existentes e remove colunas constantes.

    Remover constantes é seguro aqui porque não usa o target nem faz fit de modelo.
    Exemplo: `hashtag_followers` era sempre 0, logo não acrescentava sinal.
    """
    existing = [col for col in candidates if col in df.columns]
    missing = [col for col in candidates if col not in df.columns]

    constant = []
    if drop_constant:
        for col in existing:
            if df[col].nunique(dropna=False) <= 1:
                constant.append(col)

    selected = [col for col in existing if col not in constant]

    print(f"\n{name}")
    print("Selecionadas:", selected)
    if missing:
        print("Ignoradas por não existirem:", missing)
    if constant:
        print("Ignoradas por serem constantes:", constant)

    if not selected:
        raise ValueError(f"{name}: nenhuma feature utilizável foi selecionada.")

    return selected


# Targets são obrigatórios.
require_columns(df_profiles, [PROFILE_TARGET], "Perfis")
require_columns(df_posts, [POST_TARGET], "Posts")

prepare_target(df_profiles, PROFILE_TARGET, "Perfis")
prepare_target(df_posts, POST_TARGET, "Posts")

# Texto é útil para modelação futura, mas pode existir como Unknown se faltar.
ensure_text_columns(df_profiles, PROFILE_TEXT_COLUMNS)
ensure_text_columns(df_posts, POST_TEXT_COLUMNS)

# Conversões simples e determinísticas. Nada de scaler, encoder ou imputer fitted aqui.
prepare_numeric_candidates(df_profiles, PROFILE_NUMERIC_CANDIDATES)
prepare_numeric_candidates(df_posts, POST_NUMERIC_CANDIDATES)
prepare_categorical_candidates(df_profiles, PROFILE_CATEGORICAL_CANDIDATES)
prepare_categorical_candidates(df_posts, POST_CATEGORICAL_CANDIDATES)

PROFILE_NUMERIC_FEATURES = select_usable_features(
    df_profiles,
    PROFILE_NUMERIC_CANDIDATES,
    "Features numéricas — perfis",
)
PROFILE_CATEGORICAL_FEATURES = select_usable_features(
    df_profiles,
    PROFILE_CATEGORICAL_CANDIDATES,
    "Features categóricas — perfis",
)

POST_NUMERIC_FEATURES = select_usable_features(
    df_posts,
    POST_NUMERIC_CANDIDATES,
    "Features numéricas — posts / Modelo A",
)
POST_CATEGORICAL_FEATURES = select_usable_features(
    df_posts,
    POST_CATEGORICAL_CANDIDATES,
    "Features categóricas — posts",
)

# Segurança: o target nunca deve aparecer dentro das features.
assert PROFILE_TARGET not in PROFILE_NUMERIC_FEATURES + PROFILE_CATEGORICAL_FEATURES
assert POST_TARGET not in POST_NUMERIC_FEATURES + POST_CATEGORICAL_FEATURES

print("\nFeatures definidas com sucesso.")

## 4. Split treino/validação/teste

Usamos uma divisão **70/15/15**:

- **Treino:** usado no notebook seguinte para ajustar preprocessadores e modelos.
- **Validação:** usada para comparar modelos e afinar decisões.
- **Teste:** guardado para avaliação final imparcial.

Como o target é contínuo, criamos bins por quantis do target para tentar manter distribuições parecidas nos três conjuntos. Se os dados forem pequenos demais para isso, o notebook faz split normal.

In [ ]:
# ===============================================
# 4. Split antes de qualquer fit de preprocessador
# ===============================================


def make_stratification_bins(y, q=5):
    """Cria bins do target para stratify em regressão.

    Em regressão não há classes naturais. Usamos quantis do target como aproximação.
    Se não houver dados suficientes para bins estáveis, devolvemos None.
    """
    y = pd.Series(y).astype(float)

    for current_q in range(q, 1, -1):
        try:
            bins = pd.qcut(y, q=current_q, labels=False, duplicates="drop")
            counts = bins.value_counts(dropna=False)

            if bins.nunique(dropna=True) >= 2 and counts.min() >= 2:
                return bins
        except ValueError:
            continue

    return None


def safe_train_test_split(indices, test_size, random_state, stratify=None, label=""):
    """Executa train_test_split com fallback caso a estratificação falhe."""
    try:
        return train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )
    except ValueError as exc:
        print(f"Aviso: estratificação falhou em {label}. Split normal será usado.")
        print("Detalhe:", exc)
        return train_test_split(
            indices,
            test_size=test_size,
            random_state=random_state,
            stratify=None,
        )


def split_indices(df, target_col, test_size=0.15, val_size=0.15):
    """Devolve índices de treino, validação e teste.

    Primeiro separamos o teste.
    Depois separamos validação dentro do restante.
    Assim o teste fica protegido e não influencia decisões futuras.
    """
    y = df[target_col]
    bins = make_stratification_bins(y)

    train_val_idx, test_idx = safe_train_test_split(
        df.index,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=bins if bins is not None else None,
        label=f"teste de {target_col}",
    )

    train_val = df.loc[train_val_idx]
    train_val_bins = make_stratification_bins(train_val[target_col])

    # Queremos 15% do dataset total para validação.
    # Como o teste já tirou 15%, a validação passa a ser 15/85 dentro do restante.
    relative_val_size = val_size / (1 - test_size)

    train_idx, val_idx = safe_train_test_split(
        train_val.index,
        test_size=relative_val_size,
        random_state=RANDOM_STATE,
        stratify=train_val_bins if train_val_bins is not None else None,
        label=f"validação de {target_col}",
    )

    return pd.Index(train_idx), pd.Index(val_idx), pd.Index(test_idx)


def assert_disjoint(train_idx, val_idx, test_idx, name):
    """Garante que não há linhas repetidas entre splits."""
    train_set = set(train_idx)
    val_set = set(val_idx)
    test_set = set(test_idx)

    assert train_set.isdisjoint(val_set), f"{name}: train e validation têm interseção."
    assert train_set.isdisjoint(test_set), f"{name}: train e test têm interseção."
    assert val_set.isdisjoint(test_set), f"{name}: validation e test têm interseção."


profile_train_idx, profile_val_idx, profile_test_idx = split_indices(df_profiles, PROFILE_TARGET)
post_train_idx, post_val_idx, post_test_idx = split_indices(df_posts, POST_TARGET)

assert_disjoint(profile_train_idx, profile_val_idx, profile_test_idx, "Perfis")
assert_disjoint(post_train_idx, post_val_idx, post_test_idx, "Posts")

print("Perfis — train/validation/test:", len(profile_train_idx), len(profile_val_idx), len(profile_test_idx))
print("Posts  — train/validation/test:", len(post_train_idx), len(post_val_idx), len(post_test_idx))

In [ ]:
# ==================================================
# 4.1 Verificação das distribuições após o split
# ==================================================


def split_report(df, target_col, train_idx, val_idx, test_idx, name):
    """Mostra se os splits ficaram equilibrados em relação ao target."""
    rows = []

    for split_name, idx in [
        ("train", train_idx),
        ("validation", val_idx),
        ("test", test_idx),
    ]:
        y = df.loc[idx, target_col]
        rows.append({
            "dataset": name,
            "split": split_name,
            "n_rows": int(len(idx)),
            "target_mean": float(y.mean()),
            "target_std": float(y.std()),
            "target_min": float(y.min()),
            "target_max": float(y.max()),
        })

    return pd.DataFrame(rows)


report_profiles = split_report(
    df_profiles,
    PROFILE_TARGET,
    profile_train_idx,
    profile_val_idx,
    profile_test_idx,
    "profiles",
)

report_posts = split_report(
    df_posts,
    POST_TARGET,
    post_train_idx,
    post_val_idx,
    post_test_idx,
    "posts",
)

split_summary = pd.concat([report_profiles, report_posts], ignore_index=True)
display(split_summary)

split_summary.to_csv(METADATA_DIR / "split_summary_model_A.csv", index=False)
print("Resumo do split guardado em:", METADATA_DIR / "split_summary_model_A.csv")

## 5. Guardar apenas os dados divididos

Aqui está a correção principal.

Este notebook **não guarda modelos**, **não guarda preprocessadores** e **não guarda pipelines fitted**.

Guardamos apenas os datasets separados:

- `profiles_train.csv`
- `profiles_validation.csv`
- `profiles_test.csv`
- `posts_train.csv`
- `posts_validation.csv`
- `posts_test.csv`

Cada ficheiro inclui:

- `row_id`, para saber a linha original depois do `reset_index`;
- features selecionadas;
- colunas textuais escolhidas;
- target.

In [ ]:
# ===============================
# 5. Exportar dados de split
# ===============================


def unique_keep_order(items):
    """Remove duplicados mantendo a ordem."""
    seen = set()
    result = []

    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result


def build_export_columns(numeric_features, categorical_features, text_columns, target_col):
    """Define as colunas que seguem para o notebook de modelação."""
    return unique_keep_order(
        numeric_features
        + categorical_features
        + text_columns
        + [target_col]
    )


def export_split(df, idx, columns, output_path):
    """Guarda um split em CSV com row_id para rastreabilidade."""
    output = df.loc[idx, columns].copy()
    output.insert(0, "row_id", idx.astype(int))
    output.to_csv(output_path, index=False)
    return output.shape


profile_export_columns = build_export_columns(
    PROFILE_NUMERIC_FEATURES,
    PROFILE_CATEGORICAL_FEATURES,
    PROFILE_TEXT_COLUMNS,
    PROFILE_TARGET,
)

post_export_columns = build_export_columns(
    POST_NUMERIC_FEATURES,
    POST_CATEGORICAL_FEATURES,
    POST_TEXT_COLUMNS,
    POST_TARGET,
)

exports = {
    "profiles_train": export_split(
        df_profiles,
        profile_train_idx,
        profile_export_columns,
        TRAIN_DIR / "profiles_train.csv",
    ),
    "profiles_validation": export_split(
        df_profiles,
        profile_val_idx,
        profile_export_columns,
        VAL_DIR / "profiles_validation.csv",
    ),
    "profiles_test": export_split(
        df_profiles,
        profile_test_idx,
        profile_export_columns,
        TEST_DIR / "profiles_test.csv",
    ),
    "posts_train": export_split(
        df_posts,
        post_train_idx,
        post_export_columns,
        TRAIN_DIR / "posts_train.csv",
    ),
    "posts_validation": export_split(
        df_posts,
        post_val_idx,
        post_export_columns,
        VAL_DIR / "posts_validation.csv",
    ),
    "posts_test": export_split(
        df_posts,
        post_test_idx,
        post_export_columns,
        TEST_DIR / "posts_test.csv",
    ),
}

print("Exports concluídos:")
for name, shape in exports.items():
    print(f"- {name}: {shape}")

## 6. Guardar metadata de reprodutibilidade

A metadata não é um modelo. Serve apenas para explicar e reproduzir o split.

Isto ajuda a defender o trabalho perante o professor porque fica claro:

- qual foi o `random_state`;
- qual foi a divisão usada;
- quais features foram selecionadas;
- qual é o target;
- porque o modo dos posts é avaliação de posts existentes.

In [ ]:
# ======================================
# 6. Metadata do split e das features
# ======================================

split_indices_metadata = {
    "profiles": {
        "train": [int(i) for i in profile_train_idx],
        "validation": [int(i) for i in profile_val_idx],
        "test": [int(i) for i in profile_test_idx],
    },
    "posts": {
        "train": [int(i) for i in post_train_idx],
        "validation": [int(i) for i in post_val_idx],
        "test": [int(i) for i in post_test_idx],
    },
}

with open(METADATA_DIR / "split_indices_model_A.json", "w", encoding="utf-8") as f:
    json.dump(split_indices_metadata, f, ensure_ascii=False, indent=2)

metadata = {
    "model_mode": MODEL_MODE,
    "random_state": RANDOM_STATE,
    "splits": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "split_strategy": "70/15/15 com estratificação por quantis do target sempre que possível.",
    "important_note": "Este notebook guarda apenas dados divididos e metadata. Não guarda modelos, preprocessadores, encoders, scalers, embeddings ou pipelines fitted.",
    "profiles": {
        "target": PROFILE_TARGET,
        "numeric_features": PROFILE_NUMERIC_FEATURES,
        "categorical_features": PROFILE_CATEGORICAL_FEATURES,
        "text_columns": PROFILE_TEXT_COLUMNS,
        "export_columns": profile_export_columns,
        "n_rows": int(len(df_profiles)),
    },
    "posts": {
        "target": POST_TARGET,
        "numeric_features": POST_NUMERIC_FEATURES,
        "categorical_features": POST_CATEGORICAL_FEATURES,
        "text_columns": POST_TEXT_COLUMNS,
        "export_columns": post_export_columns,
        "n_rows": int(len(df_posts)),
        "interpretation": "Modelo A: avaliação de posts existentes. Inclui reactions, comments e time_spent_days de forma intencional.",
        "not_for": "Não apresentar como previsão antes da publicação.",
        "removed_or_ignored_features": [
            "hashtag_followers quando constante/sem informação útil",
            "features inexistentes na versão processada do dataset",
            "features constantes",
        ],
    },
    "exports": {
        "train_dir": str(TRAIN_DIR),
        "validation_dir": str(VAL_DIR),
        "test_dir": str(TEST_DIR),
        "metadata_dir": str(METADATA_DIR),
    },
}

with open(METADATA_DIR / "feature_split_metadata_model_A.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Metadata guardada em:", METADATA_DIR)
print("- split_indices_model_A.json")
print("- feature_split_metadata_model_A.json")

## Conclusão do notebook 02

Este notebook criou os datasets separados para a fase seguinte de modelação.

Frase útil para o relatório/apresentação:

> Foi usado um split 70/15/15 para separar treino, validação e teste. O treino será usado para ajustar modelos e preprocessadores, a validação para comparar modelos e afinar decisões, e o teste para avaliação final imparcial. Como o problema é de regressão, foi usada estratificação por quantis do target sempre que possível, para manter distribuições semelhantes dos scores nos três conjuntos.

Outra frase importante:

> No caso dos posts, o modo escolhido foi avaliação de posts existentes. Por isso, variáveis como reações, comentários e tempo desde publicação foram mantidas como features. Este modo não deve ser confundido com previsão antes da publicação, onde essas variáveis teriam de ser removidas.

Agora o próximo notebook pode ser `03_model_training.ipynb`, onde aí sim faz sentido ajustar preprocessadores, treinar modelos e guardar artefactos de modelação.